In [5]:
import json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [ ]:
#helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 500,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [3]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing a task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task"
  }
]

Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex.

Focus on tasks that do not require writing much code.

Please generate 3 objects.
"""

    messages = []

    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")

    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [6]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket ARN (e.g., 'arn:aws:s3:::my-bucket'). Return the region or None if not found."}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'."}, {'task': "Write a regex pattern that matches valid AWS access key IDs (format: starts with 'AKIA' followed by 20 uppercase alphanumeric characters)."}]


In [7]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [8]:
#evaluation pipeline
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [19]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)

    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [20]:
from statistics import mean

def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])

    print(f"Average score: {average_score}")

    return results

In [21]:
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.166666666666667


In [22]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket ARN Region Extractor\n\nHere's a comprehensive solution:\n\n```python\ndef extract_region_from_s3_arn(arn):\n    \"\"\"\n    Extract the AWS region from an S3 bucket ARN.\n    \n    S3 bucket ARNs have the format:\n    - arn:aws:s3:::bucket-name (global, no region)\n    - arn:aws:s3:region:account-id:bucket/bucket-name (object ARN with region)\n    \n    Args:\n        arn (str): The S3 bucket or object ARN\n        \n    Returns:\n        str or None: The AWS region if found, None otherwise\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        return None\n    \n    # Split the ARN by colons\n    parts = arn.split(':')\n    \n    # Valid S3 ARN format: arn:aws:s3:[region]:[account-id]:bucket[/object-path]\n    # S3 bucket ARNs typically don't have regions, but object ARNs might\n    if len(parts) >= 6:\n        # For object ARNs: arn:aws:s3:region:account-id:bucket/key\n        region = parts[3]\n        if region:  # Non-empty region s

In [18]:
def grade_by_model(test_case, output):
    #потому что нам нужно реально подставить
    eval_prompt = f""" 
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])

    return json.loads(eval_text)